In [1]:
import os
import sys
import datetime
import pandas as pd
import time
import traceback
import _pickle as pickle

# upper_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
upper_dir = '/home/trade_core' # TEST
sys.path.append(upper_dir)
from loggers.logger import KimpBotLogger
from exchange_plugin.upbit_plug import UserUpbitAdaptor
from exchange_plugin.binance_plug import UserBinanceAdaptor
from etc.redis_connector.redis_connector import InitRedis

In [2]:
binance_access_key = '4PFfYsObYyUaMlk7m6tT9qIl8X8P3WCUsyu1lEyZ4h50VfTMPIQCNdL9eOJCl3Lb'
binance_secret_key = '011Z1W9p4425AZgtCNbs5L1d77ehZFNmBIwT0pz1SJGP7EiOlPILfWBglbVsxMcK'

upbit_access_key = "x2AKLfsRtAgRxFjQ7p9TZTAg6E1SveoqfNfD8MK8"
upbit_secret_key = "svEran52QdsUyJdxAPYoTgFT2MXsHc5ZiKsKPxTL"

okx_access_key = "fcb66a6e-0443-4743-8d9b-61caf7eab662"
okx_secret_key = "0029E09874B38F5AC7E68E9DFC348667"
okx_passphrase = "121431Rn!!"

bybit_access_key = "S3YKBbD0kz1WwcfqZD"
bybit_secret_key = "390M6dKJ9J0uEK7vXYAl3qxVCh944tRrzHah"

In [36]:
class UserExchangeAdaptor:
    def __init__(self, logging_dir=None):
        self.logger = KimpBotLogger("integrated_plug", logging_dir=logging_dir).logger
        self.logger.info('integrated_plug_logger init')
        self.exchange_adaptor_dict = {}
        self.exchange_adaptor_dict["UPBIT"] = UserUpbitAdaptor(logging_dir=logging_dir)
        self.exchange_adaptor_dict["BINANCE"] = UserBinanceAdaptor(logging_dir=logging_dir)
        # redis connection to read info_dict
        self.local_redis_client = InitRedis(host='localhost', port=6379, db=0, passwd=None)

    def check_api_key(self, exchange, access_key, secret_key, futures=False):
        """FUTURES includes USD_M, COIN_M"""
        if exchange.upper() not in self.exchange_adaptor_dict:
            raise Exception(f'exchange {exchange} not supported')
        return self.exchange_adaptor_dict[exchange].check_api_key(access_key, secret_key, futures)
    
    def get_position(self, exchange, access_key, secret_key, market_type):
        """SPOT position columns: ['asset', 'free', 'locked']
        USD_M position columns: []
        """

        if exchange.upper() not in self.exchange_adaptor_dict:
            raise Exception(f'exchange {exchange} not supported')
        exchange_adaptor = self.exchange_adaptor_dict[exchange]
        if exchange == "UPBIT":
            position_df = exchange_adaptor.get_balance(access_key, secret_key, market_type)
        elif exchange == "BINANCE":
            position_df = exchange_adaptor.all_position_information(access_key, secret_key, market_type)
            info_df = pickle.loads(self.local_redis_client.get_data(f'TRADE_CORE|{exchange.lower()}_{market_type.lower()}_info_df'))
            position_df = position_df.merge(info_df[['symbol','base_asset']], how='left', on='symbol')
            position_df = position_df.rename(columns={"positionAmt":"qty", "marginType":"margin_type", "entryPrice":"entry_price", "liquidationPrice":"liquidation_price"})
            position_df["ROI"] = position_df.apply(lambda x: (x['entry_price']-x['markPrice'])/x['markPrice']*x['leverage']*100 if x['qty']<0 else 
                                                   (x['markPrice']-x['entry_price'])/x['entry_price']*['leverage']*100, axis=1)
        else:
            raise Exception(f'exchange {exchange} not supported')
        return position_df

In [37]:
user_exchange_adaptor = UserExchangeAdaptor()

2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,890 - integrated_plug - INFO - integrated_plug_logger init
2024-02-17 12:04:16,895 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-17 12:04:16,895 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-17 12:04:16,895 - user_upbit_plug - INFO - user_upbit_plug_logger started.
2024-02-17 12:04:16,895 - user_upbit_plu

In [11]:
pickle.loads(user_exchange_adaptor.local_redis_client.get_data('TRADE_CORE|binance_usd_m_info_df'))

,symbol,pair,contractType,deliveryDate,onboardDate,status,maintMarginPercent,requiredMarginPercent,base_asset,quote_asset,...,underlyingSubType,settlePlan,triggerProtect,liquidationFee,marketTakeBound,maxMoveOrderLimit,filters,orderTypes,timeInForce,perpetual
0,BTCUSDT,BTCUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,BTC,USDT,...,[PoW],0,0.05,0.0125,0.05,10000,"[{'maxPrice': '4529764', 'tickSize': '0.10', '...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
1,ETHUSDT,ETHUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,ETH,USDT,...,[Layer-1],0,0.05,0.0125,0.05,10000,"[{'minPrice': '39.86', 'tickSize': '0.01', 'ma...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
2,BCHUSDT,BCHUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,BCH,USDT,...,[PoW],0,0.05,0.0150,0.05,10000,"[{'tickSize': '0.01', 'minPrice': '13.93', 'ma...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
3,XRPUSDT,XRPUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,XRP,USDT,...,[Payment],0,0.05,0.0125,0.05,10000,"[{'maxPrice': '100000', 'minPrice': '0.0143', ...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
4,EOSUSDT,EOSUSDT,PERPETUAL,4133404800000,1569398400000,TRADING,2.5,5.0,EOS,USDT,...,[Layer-1],0,0.10,0.0100,0.10,10000,"[{'minPrice': '0.111', 'maxPrice': '100000', '...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,RONINUSDT,RONINUSDT,PERPETUAL,4133404800000,1707193800000,TRADING,2.5,5.0,RONIN,USDT,...,[Layer-1],0,0.15,0.0150,0.15,10000,"[{'tickSize': '0.001000', 'minPrice': '0.00100...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
274,DYMUSDT,DYMUSDT,PERPETUAL,4133404800000,1707289200000,TRADING,2.5,5.0,DYM,USDT,...,[Infrastructure],0,0.15,0.0150,0.15,10000,"[{'minPrice': '0.001000', 'filterType': 'PRICE...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
275,SUIUSDC,SUIUSDC,PERPETUAL,4133404800000,1707375600000,TRADING,2.5,5.0,SUI,USDC,...,[USDC],0,0.15,0.0150,0.15,10000,"[{'filterType': 'PRICE_FILTER', 'minPrice': '0...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True
276,OMUSDT,OMUSDT,PERPETUAL,4133404800000,1707832800000,TRADING,2.5,5.0,OM,USDT,...,[DeFi],0,0.15,0.0150,0.15,10000,"[{'filterType': 'PRICE_FILTER', 'tickSize': '0...","[LIMIT, MARKET, STOP, STOP_MARKET, TAKE_PROFIT...","[GTC, IOC, FOK, GTX, GTD]",True


In [41]:
user_exchange_adaptor.get_position('BINANCE', binance_access_key, binance_secret_key, 'USD_M')#[['base_asset','qty','liquidation_price','ROI']]

,symbol,qty,entry_price,breakEvenPrice,markPrice,unRealizedProfit,liquidation_price,leverage,maxNotionalValue,margin_type,isolatedMargin,isAutoAddMargin,positionSide,notional,isolatedWallet,updateTime,isolated,adlQuantile,base_asset,ROI
0,ETCUSDT,-291.78,24.797714,22.401123,26.417961,-472.755775,122.686163,4.0,10000000.0,cross,0.00000000,false,BOTH,-7708.23266933,0,1708100728008,False,1,ETC,-24.532512
1,STXUSDT,-4944.0,1.464272,1.46354,2.521553,-5227.196226,8.118061,4.0,2400000.0,cross,0.00000000,false,BOTH,-12466.55921856,0,1705623903014,False,1,STX,-167.718971
2,SOLUSDT,-42.0,84.171,84.128914,109.504,-1063.986,780.593383,4.0,20000000.0,cross,0.00000000,false,BOTH,-4599.16800000,0,1706055101469,False,1,SOL,-92.537259
3,GMTUSDT,-15175.0,0.2373,0.237181,0.273,-541.7475,2.1143,4.0,2000000.0,cross,0.00000000,false,BOTH,-4142.77500000,0,1707269040692,False,1,GMT,-52.307692
4,XRPUSDT,-10227.0,0.5634,0.563118,0.5549,86.9295,3.302254,4.0,12000000.0,cross,0.00000000,false,BOTH,-5674.96230000,0,1708100262308,False,2,XRP,6.127230
5,SEIUSDT,-18497.0,0.851325,0.730756,0.9382,-1606.935754,2.408298,4.0,1200000.0,cross,0.00000000,false,BOTH,-17353.88540000,0,1708099217904,False,1,SEI,-37.039215
6,LSKUSDT,-2442.0,1.467641,1.466907,1.3674,244.789279,12.645817,4.0,500000.0,cross,0.00000000,false,BOTH,-3339.19080000,0,1707209882261,False,3,LSK,29.323188
7,VETUSDT,-151592.0,0.047553,0.04753,0.04439,479.527942,0.228678,4.0,5000000.0,cross,0.00000000,false,BOTH,-6729.16888000,0,1708100362647,False,3,VET,28.504444
